# ⚡ Lightning Data Module

This notebook shows how to use the various data modules defined in this project.

## Setup

---

Let's install some necessary dependencies and set global variables.

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
# Autoroot
import autorootcwd

In [ ]:
import cv2 as cv

# Imports
import numpy as np
import torch
import torchvision.transforms.v2 as transforms
from matplotlib import pyplot as plt
from torch.utils.data import DataLoader
from tqdm import tqdm

# Local modules
from src.data.components.paired import PairedDataset
from src.utils.utils import undo_transforms

## ImagePairDataset

---

This data module loads

In [ ]:
# Initialise data module
augmentations = {
    "flip": True,
    "rotate": True,
    "blur": True,
    "brightness": True
}

dataset = PairedDataset(data_dir="data/processed/", patch_size=256, augmentations=augmentations)
print(f"✅ Loaded ImagePairDataset with {len(dataset)} samples")

In [ ]:
# Inspect sample
film, digital = dataset[0]
print(f"Film image: {film.shape}, Digital image {digital.shape}")

# Show sample
fig, axs = plt.subplots(ncols=2, figsize=(15, 10))
axs[0].imshow(undo_transforms(film).numpy())
axs[1].imshow(undo_transforms(digital).numpy())
axs[0].set_title("Film")
axs[1].set_title("Digital");

In [ ]:
# Create dataloader
batch_size = 4
dataloader = DataLoader(dataset, batch_size=batch_size)

film_batch, digital_batch = next(iter(dataloader))
print(f"Film Batch: {film_batch.shape}, Digital batch: {digital_batch.shape}")

# Show sample
fig, axs = plt.subplots(nrows=batch_size, ncols=2, figsize=(16, 8 * batch_size))
for i in range(batch_size):
    axs[i, 0].imshow(undo_transforms(film_batch[i]).numpy())
    axs[i, 1].imshow(undo_transforms(digital_batch[i]).numpy())
    axs[i, 0].set_title("Film")
    axs[i, 1].set_title("Digital");

In [ ]:
# Lightning data module
from src.data.paired_module import PairedDataModule

datamodule = PairedDataModule(dataset)
datamodule.prepare_data()
datamodule.setup()
train_loader = datamodule.train_dataloader()
print(f"Train batches: {len(train_loader)}")
batch = next(iter(train_loader))
print(batch[0].shape, batch[1].shape)